# Notebook 02 — Deep-learning methods for the comparison study

**What this adds to the study:** the local scripts compare classical models on hand-made colour
and texture features. This notebook adds the deep-learning side, scored on **exactly the same tests**:

1. **Pretrained networks as feature extractors** (ResNet-50, EfficientNet-B0, ConvNeXt-Tiny, DINOv2-small).
   Each conjunctiva image goes through the network once; we keep its internal description (a few
   hundred numbers) and save it. The local script then trains the same 8 classical models on these.
2. **Fine-tuned networks** (EfficientNet-B0, ConvNeXt-Tiny): the whole network is trained on our images,
   inside the same train/test splits, with near-duplicate copies of an eye kept on one side.

**Outputs** (`/kaggle/working`): `embeddings_<network>.npz` and `finetune_scores_<network>.csv`.
Download them into `MajorProject/results/study/`, then run `python src/study/exp_models.py` on the laptop.

**Run on Kaggle:** add the private dataset `anemia-data` (version 2, which contains `groups.csv`),
Accelerator = GPU T4, Internet = On (needed to download the pretrained weights), then Save & Run All.
Expect about 40–70 minutes.

In [ ]:
import os, glob, time, random, warnings
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn
import timm
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
QUICK = os.environ.get("ANEMIA_QUICK", "0") == "1"          # self-test switch; leave alone on Kaggle
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if os.environ.get("ANEMIA_DATA"): hits = [os.path.join(os.environ["ANEMIA_DATA"], name)] + hits
    assert hits, f"{name} not found: add version 2 of the anemia dataset (it contains groups.csv)"
    return hits[0]

DATA = os.path.dirname(find("labels.csv"))     # folder with labels.csv, conj/, eye/, mask/
GROUPS = find("groups.csv")                    # near-duplicate groups (may sit in a different folder)
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("ANEMIA_OUT", "outputs")
os.makedirs(OUT, exist_ok=True)
print("data:", DATA, "| device:", DEV)

## 1. Data and the tests

Same rules as the rest of the study:
- Ghana (CP-AnemiC): only the 392 images that are not exact copies; near-duplicate copies (same eye,
  mirrored or re-cropped) share a `group_strict` id and are never split between training and testing.
- India and Italy (Eyes-Defy-Anemia) are always tested **separately**.
- Cross-population tests: train on one population, test on another.

In [ ]:
lab = pd.read_csv(os.path.join(DATA, "labels.csv")).merge(pd.read_csv(GROUPS), on="subject_id")
lab = lab[lab.hb.notna()].reset_index(drop=True)
y = lab.anemic.values.astype(int); site = lab.site.values; G = lab.group_strict.values
S = {s: np.where(site == s)[0] for s in ("Ghana", "India", "Italy")}
GH = S["Ghana"][lab.usable.values[S["Ghana"]] == 1]; ED = np.r_[S["India"], S["Italy"]]
print(lab.groupby("site").agg(n=("hb", "size"), anaemic=("anemic", "sum"), usable=("usable", "sum")))

def splits():
    """Every (test name, train rows, test rows) used for fine-tuning. One repeat of 5-fold CV."""
    out = []
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(GH, y[GH], G[GH]):
        out.append(("Ghana (copies grouped)", GH[a], GH[b]))
    for s in ("India", "Italy"):
        idx = S[s]
        for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(idx, y[idx]):
            out.append((s, idx[a], idx[b]))
    out += [("Ghana → India", GH, S["India"]), ("Ghana → Italy", GH, S["Italy"]), ("India+Italy → Ghana", ED, GH),
            ("India → Italy", S["India"], S["Italy"]), ("Italy → India", S["Italy"], S["India"])]
    return out

## 2. Turning a cut-out into a network input

The cut-out has a transparent background. We place it on black, pad it to a square so the shape is not
stretched, and resize to 224×224. **No colour changes are applied**, because colour is the signal.

In [ ]:
SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

def load_img(path):
    im = cv2.imread(os.path.join(DATA, path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = (im / 257).astype(np.uint8)
    if im.shape[2] == 4:
        a = (im[:, :, 3:4] > 127).astype(np.float32)
        if a.mean() > 0.9: a = (im[:, :, :3].min(2, keepdims=True) < 245).astype(np.float32)
        im = (im[:, :, :3] * a).astype(np.uint8)
    h, w = im.shape[:2]; s = max(h, w)
    sq = np.zeros((s, s, 3), np.uint8); sq[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = im
    return cv2.resize(sq, (SIZE, SIZE), interpolation=cv2.INTER_AREA)[:, :, ::-1]

IMGS = np.stack([load_img(p) for p in lab.conj_path])
print("images", IMGS.shape)

def to_tensor(batch):
    x = (batch.astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(x.transpose(0, 3, 1, 2).copy())

## 3. Pretrained networks as fixed feature extractors

Each network was trained by others on ImageNet (1.2 million everyday photos). We do **not** train it here;
we only read out its last internal layer. Each image is also flipped and the two readings averaged.

In [ ]:
BACKBONES = {"resnet50": "resnet50.a1_in1k", "efficientnet_b0": "efficientnet_b0.ra_in1k",
             "convnext_tiny": "convnext_tiny.fb_in1k", "dinov2_small": "vit_small_patch14_dinov2.lvd142m"}
if QUICK: BACKBONES = {"efficientnet_b0": "efficientnet_b0.ra_in1k"}

@torch.no_grad()
def embed(name):
    m = timm.create_model(name, pretrained=not QUICK, num_classes=0, img_size=SIZE) if "dinov2" in name \
        else timm.create_model(name, pretrained=not QUICK, num_classes=0)
    m = m.eval().to(DEV); out = []
    for i in range(0, len(IMGS), 32):
        x = to_tensor(IMGS[i:i + 32]).to(DEV)
        out.append(((m(x) + m(torch.flip(x, dims=[3]))) / 2).float().cpu().numpy())
    return np.concatenate(out)

for short, name in BACKBONES.items():
    t = time.time(); E = embed(name)
    np.savez_compressed(os.path.join(OUT, f"embeddings_{short}.npz"), subject_id=lab.subject_id.values, emb=E.astype(np.float32))
    print(f"{short}: {E.shape}  {time.time() - t:.0f}s")

## 4. Fine-tuning a whole network

Here the network's weights are updated on our images. For each test the model starts again from the
ImageNet weights, sees only that test's training images, and then scores the test images.

Settings: 12 epochs, AdamW, learning rate 3e-4 with cosine decay, batch 16, class-weighted loss.
Augmentation is **geometric only** (flips, small rotation and zoom): changing colour or brightness would
teach the network to ignore exactly the redness we are trying to measure.

In [ ]:
EPOCHS = 2 if QUICK else 12

def augment(img, rng):
    if rng.random() < 0.5: img = img[:, ::-1]
    if rng.random() < 0.5: img = img[::-1]
    ang = rng.uniform(-15, 15); sc = rng.uniform(0.9, 1.1)
    M = cv2.getRotationMatrix2D((SIZE / 2, SIZE / 2), ang, sc)
    return cv2.warpAffine(np.ascontiguousarray(img), M, (SIZE, SIZE), borderValue=0)

def finetune_and_score(name, tr, te, seed=SEED):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    m = timm.create_model(name, pretrained=not QUICK, num_classes=1).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-2)
    steps = EPOCHS * int(np.ceil(len(tr) / 16)); sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, total_steps=steps, pct_start=0.15)
    pos = y[tr].mean(); lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - pos) / max(pos, 1e-3)], device=DEV))
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    for ep in range(EPOCHS):
        m.train(); order = rng.permutation(tr)
        for i in range(0, len(order), 16):
            b = order[i:i + 16]
            if len(b) < 2: continue
            x = to_tensor(np.stack([augment(IMGS[k], rng) for k in b])).to(DEV)
            t = torch.tensor(y[b], dtype=torch.float32, device=DEV)
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                loss = lossf(m(x).squeeze(1), t)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    m.eval(); s = []
    with torch.no_grad():
        for i in range(0, len(te), 32):
            x = to_tensor(IMGS[te[i:i + 32]]).to(DEV)
            s.append(((torch.sigmoid(m(x)) + torch.sigmoid(m(torch.flip(x, dims=[3])))) / 2).squeeze(1).float().cpu().numpy())
    return np.concatenate(s)

FT = {"efficientnet_b0": "efficientnet_b0.ra_in1k", "convnext_tiny": "convnext_tiny.fb_in1k"}
if QUICK: FT = {"efficientnet_b0": "efficientnet_b0.ra_in1k"}
SPL = splits() if not QUICK else [s for s in splits() if s[0] in ("India", "Ghana → India")][:2]
for short, name in FT.items():
    rows = []; t0 = time.time()
    for test, tr, te in SPL:
        s = finetune_and_score(name, tr, te)
        rows += [{"test": test, "subject_id": lab.subject_id[k], "y": y[k], "score": v} for k, v in zip(te, s)]
        print(f"{short:16s} {test:24s} done  ({time.time() - t0:.0f}s)", flush=True)
    R = pd.DataFrame(rows); R.to_csv(os.path.join(OUT, f"finetune_scores_{short}.csv"), index=False)
    print(short, {t: round(roc_auc_score(g.y, g.score), 3) for t, g in R.groupby("test") if g.y.nunique() == 2})

## 5. Done

Download everything in the **Output** tab into `MajorProject/results/study/`. On the laptop, run
`python src/study/exp_models.py` (adds the four embedding feature sets to the comparison) and
`python src/study/report.py` (adds the fine-tuned networks and rebuilds the tables and plots).

In [ ]:
print(sorted(os.listdir(OUT)))